# DYN heatmap — per-sample figure/table sets

One heatmap per sample: taxa x 6 tools, one panel per platform (PacBio / ONT-Qiagen / ONT-Zymo).
Holding the sample fixed is what makes the figure interpretable — every difference across a row is
method variation, not biology. **Deliberately no cohort/averaged heatmap**: averaging abundances
across biologically distinct samples confounds biology with method and produces a 'mean community'
that describes no real sample. Cohort-level questions (agreement, variance partition, reproducibility)
are answered in `dyn_cohort.ipynb` with agreement statistics instead.

Outputs: `<TS>/dyn-heatmap/<Sample>/{figures,tables}`.
Rows = union of each library's top-K species. Abundance split fixed at 5%.

## Import

In [1]:
# Make the `utils` helper package importable regardless of CWD.
# Layout: bakeoff/scripts/analysis/utils/{parser,util}.py
import sys
from pathlib import Path as _PathBootstrap
_HERE = _PathBootstrap.cwd()
for _cand in [_HERE / 'scripts' / 'analysis',  # CWD = bakeoff/
              _HERE,                            # CWD = bakeoff/scripts/analysis/
              _HERE / 'bakeoff' / 'scripts' / 'analysis']:
    if (_cand / 'utils' / '__init__.py').exists() and str(_cand) not in sys.path:
        sys.path.insert(0, str(_cand))
        break

import os
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr
from scipy.spatial.distance import braycurtis, cosine
from scipy.cluster.hierarchy import linkage

from utils.util import _clean_name

## Configuration

In [2]:
# Global inputs (DYN heatmap, cache-driven)
# Reads the cohort cache produced by bakeoff/scripts/analysis/dyn_prep.py — run that first.
# NOTE: this notebook processes EVERY DYN sample present on all three platforms; the sample
# list is discovered at runtime by dyn_samples(). There is no single-sample selector — set
# DISPLAY_SAMPLE (further down) only to choose which one renders inline.

from pathlib import Path
import matplotlib.pyplot as plt

MIN_ABUNDANCE = 0.1   # percent: detection floor / colour floor
RANKS = ["species", "genus"]

# Platforms shown as panels, left -> right
PANEL_COHORTS = ["pacbio", "ont_qiagen", "ont_zymo"]
PANEL_LABELS  = {"pacbio": "PacBio", "ont_qiagen": "ONT Qiagen", "ont_zymo": "ONT Zymo"}

# Cohort cache (output of dyn_prep.py — most recent timestamped run)
from utils.alpha_div import find_latest_cohort_cache
from utils.results   import find_latest_dyn_prep_ts, save_csv, save_fig

SAVE = True    # write figures/tables for ALL samples to OUT_BASE
RESULTS_ROOT = Path("/home/Users/pacbio_bakeoff/bakeoff/results")
TS           = find_latest_dyn_prep_ts(RESULTS_ROOT)  # or pin: TS = "YYYYMMDD_HHMMSS"
CACHE_DIR    = find_latest_cohort_cache(RESULTS_ROOT)


def cache_tsv_path(cohort: str, tool: str, db_mode: str, rank: str) -> Path:
    """Path to the cohort cache TSV for one (cohort, tool, db_mode, rank)."""
    return CACHE_DIR / f"{cohort}_{tool}_{db_mode}_{rank}.tsv"


# Outputs: <TS>/dyn-heatmap/<Sample>/{figures,tables} — one directory per sample.
OUT_BASE = RESULTS_ROOT / TS / "dyn-heatmap"


def sample_dirs(sample):
    f, t = OUT_BASE / sample / "figures", OUT_BASE / sample / "tables"
    f.mkdir(parents=True, exist_ok=True); t.mkdir(parents=True, exist_ok=True)
    return f, t


# Figure parameters
FIG_DPI = 300
FIG_FORMAT = "png"

# Tool order = column order within each platform panel
TOOL_ORDER = ["Centrifuge", "Centrifuger", "Kraken2", "Ganon2", "Sourmash", "Sylph"]

# Every tool uses the shared unified reference, so the DB is held fixed and differences
# between columns reflect the algorithm rather than the database.
SELECTED_DB_PER_TOOL = {
    "Kraken2":     "unified",
    "Centrifuge":  "unified",
    "Centrifuger": "unified",
    "Sourmash":    "unified",
    "Sylph":       "unified",
    "Ganon2":      "unified",   # unified rebuilt after a major bug fix
}

# Map display-cased tool name -> cache filename token
_TOOL_NAME_IN_CACHE = {
    "Kraken2":     "Kraken2",
    "Centrifuge":  "Centrifuge",
    "Centrifuger": "Centrifuger",
    "Sourmash":    "sourmash",
    "Sylph":       "sylph",
    "Ganon2":      "ganon2",
}

# ---- Per-platform taxa pool ----
# For each platform, a taxon enters the pool if >= CONSENSUS_MIN_SUPPORT tools on that platform
# report it at >= PRESENCE_MIN_ABUND. Kept permissive (=1); the row rules further down
# (top-K union, >=N-tool agreement) do the actual selection.
PRESENCE_MIN_ABUND    = MIN_ABUNDANCE
CONSENSUS_MIN_SUPPORT = 1

# ---- Two-range abundance colour scale ----
# Low range [ABUND_FLOOR, ABUND_SPLIT) and high range [ABUND_SPLIT, ABUND_CEIL] each get a
# distinct hue with a log-scaled light->dark gradient. ABUND_SPLIT is also drawn as a black
# line on the colourbar and always gets a tick label.
ABUND_FLOOR  = MIN_ABUNDANCE   # 0.1%
ABUND_SPLIT  = 5.0             # % boundary between "low" (green) and "high" (orange)
ABUND_CEIL   = 100.0
LOW_RAMP     = ["#e5f5e0", "#a1d99b", "#238b45"]   # greens  -> low  taxa
HIGH_RAMP    = ["#fdd0a2", "#fd8d3c", "#a63603"]   # oranges -> high taxa
NO_HIT_COLOR = "#999DA0"

print(f"CACHE_DIR    : {CACHE_DIR if CACHE_DIR.exists() else '(missing — run dyn_prep.py first)'}")
print(f"Output base  : {OUT_BASE}   (SAVE={SAVE})")
print(f"Platforms    : {PANEL_COHORTS}")

# Existence sanity for every platform × tool × rank cache TSV
missing = []
for cohort in PANEL_COHORTS:
    for tool, db_mode in SELECTED_DB_PER_TOOL.items():
        for rank in RANKS:
            p = cache_tsv_path(cohort, _TOOL_NAME_IN_CACHE[tool], db_mode, rank)
            if not p.exists():
                missing.append(p)
if missing:
    print(f"\n[WARN] {len(missing)} missing cohort-cache TSVs:")
    for p in missing[:10]:
        print("  -", p)
    if len(missing) > 10:
        print(f"  ... (+{len(missing) - 10} more)")
else:
    print("\n✓ All required cohort-cache TSVs present.")

print(f"\n✓ Configuration loaded")
print(f"  - Ranks: {RANKS}")
print(f"  - detection floor: {MIN_ABUNDANCE}%   colour split: {ABUND_SPLIT}%")


CACHE_DIR    : /home/Users/pacbio_bakeoff/bakeoff/results/prepared/20260713_015342_dyn/dyn-prep/tables/cohorts
Output base  : /home/Users/pacbio_bakeoff/bakeoff/results/20260713_015342_dyn/dyn-heatmap   (SAVE=True)
Platforms    : ['pacbio', 'ont_qiagen', 'ont_zymo']

✓ All required cohort-cache TSVs present.

✓ Configuration loaded
  - Ranks: ['species', 'genus']
  - detection floor: 0.1%   colour split: 5.0%


## Shared machinery

In [3]:
# Shared machinery: loader, taxa selection, column order, matrix builder.

SYNONYM_TAXID_MAP = {1578: 2742598}

_KEY_COLS = ["key", "taxid", "name", "abundance"]

def _load_from_cache(cohort, tool, db_mode, rank, sample_id_core):
    """Return DataFrame(key, taxid, name, abundance) for one biological sample."""
    cache_tool = _TOOL_NAME_IN_CACHE[tool]
    p = cache_tsv_path(cohort, cache_tool, db_mode, rank)
    if not p.exists():
        return pd.DataFrame(columns=_KEY_COLS)

    df = pd.read_csv(p, sep="\t", usecols=["sample_id_core", "taxid", "name", "norm_abund"])
    df = df[df["sample_id_core"].astype(str) == str(sample_id_core)]
    if df.empty:
        return pd.DataFrame(columns=_KEY_COLS)

    df = df.rename(columns={"norm_abund": "abundance"})
    df["taxid"] = pd.to_numeric(df["taxid"], errors="coerce").fillna(0).astype(int)
    df["taxid"] = df["taxid"].map(lambda t: SYNONYM_TAXID_MAP.get(t, t))
    df["name"] = df["name"].astype(str)
    df["abundance"] = pd.to_numeric(df["abundance"], errors="coerce").fillna(0.0)

    # Drop generic non-taxon rows that confuse a per-taxon view
    df = df[~df["name"].str.lower().isin(["unclassified", "root", "unknown"])].copy()

    df["key"] = np.where(df["taxid"] > 0, df["taxid"].astype(str), "name:" + df["name"])
    # Collapse duplicates on the canonical key (name variants, synonym-merged taxids)
    df = df.groupby("key", as_index=False).agg(
        taxid=("taxid", "first"), name=("name", "first"), abundance=("abundance", "sum"))
    return df[_KEY_COLS]


def build_taxa_selection(
    cohort: str,
    sample_id_core: str,
    selected_db_per_tool: dict,
    ranks: list[str],
    k: int = 1,
    min_abund: float = 0.1,
) -> tuple[dict, dict]:
    """Per-rank taxa sets from one cohort's own tools.

    Presence definition: abundance >= min_abund (percent units). Keyed on canonical taxid.
    """
    whitelist_taxa: dict[str, set] = {}
    support_counts: dict[str, pd.DataFrame] = {}

    for rank in ranks:
        support: dict[str, set] = {}
        key_name: dict[str, str] = {}
        parsed_any = 0

        for tool, db_mode in selected_db_per_tool.items():
            tool_db = f"{tool}_{db_mode}"
            df = _load_from_cache(cohort, tool, db_mode, rank, sample_id_core)
            if df.empty:
                print(f"[WARN] {cohort} cache empty for {tool_db} / {rank} / {sample_id_core}")
                continue

            parsed_any += 1
            present = df.loc[df["abundance"] >= float(min_abund)]
            for key, nm in zip(present["key"], present["name"]):
                support.setdefault(key, set()).add(tool_db)
                key_name.setdefault(key, nm)

        if parsed_any == 0:
            print(f"[WARN] No {cohort} cache rows parsed for rank={rank}")
            whitelist_taxa[rank] = set()
            support_counts[rank] = pd.DataFrame(columns=["key", "name", "support_count", "supporting_tool_dbs"])
            continue

        rows = []
        for key, tdb_set in support.items():
            rows.append({
                "key":                 key,
                "name":                key_name.get(key, key),
                "support_count":       len(tdb_set),
                "supporting_tool_dbs": ",".join(sorted(tdb_set)),
            })
        counts = pd.DataFrame(rows).sort_values(["support_count", "name"], ascending=[False, True]).reset_index(drop=True)

        whitelist_taxa[rank] = set(counts.loc[counts["support_count"] >= int(k), "key"].tolist())
        support_counts[rank] = counts

        print(f"[taxa selection] {cohort} rank={rank}: parsed {parsed_any} tool_db files, "
              f"{len(whitelist_taxa[rank])} taxa with support >= {k}")

    return whitelist_taxa, support_counts


SELECTED_TOOLDBS = [
    f"{tool}_{SELECTED_DB_PER_TOOL[tool]}"
    for tool in TOOL_ORDER
    if tool in SELECTED_DB_PER_TOOL
]

def create_abundance_matrix(tools_data: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    rows = canonical key (taxid, or name for unmapped taxa)
    cols = tool_db
    values = abundance (%)
    NaN means 'not reported by this tool_db'
    """
    mats = []
    for tool_db, df in tools_data.items():
        if df is None or df.empty:
            continue

        d = df.copy()
        if "key" not in d.columns or "abundance" not in d.columns:
            raise ValueError(f"{tool_db} missing required columns: key/abundance")

        d["abundance"] = pd.to_numeric(d["abundance"], errors="coerce").fillna(0.0)

        # aggregate duplicates (just in case)
        d = d.groupby("key", as_index=True)["abundance"].sum().to_frame(tool_db)
        mats.append(d)

    if not mats:
        return pd.DataFrame()

    return pd.concat(mats, axis=1)


In [4]:
# Shared machinery: colour scale, per-sample panel data, top-K union ranking, renderer.
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle

# PANEL_COHORTS / PANEL_LABELS come from the Configuration cell.
PER_TOOL_TOPK = 20      # each library's top-K species; union -> plotted rows
                        # (species union: median 32 / max 38 rows -> auto-compresses
                        #  to ~0.20 in/row on a 9 in page; K=8 gives max 32 with margin)
LETTER_W, LETTER_H = 6.5, 9.0    # US Letter text area (in)
ROW_H, OVERHEAD_H  = 0.11, 1.4   # row pitch (in); 6pt labels need >~0.11

def _two_range_scale(vmax=None):
    """(cmap, norm, floor, ceil) for the low/high two-range log color scale."""
    floor = float(globals().get("ABUND_FLOOR", 0.1))
    split = float(globals().get("ABUND_SPLIT", 1.0))
    ceil  = float(vmax if vmax is not None else globals().get("ABUND_CEIL", 100.0))
    low_ramp  = globals().get("LOW_RAMP",  ["#e5f5e0", "#a1d99b", "#238b45"])
    high_ramp = globals().get("HIGH_RAMP", ["#fdd0a2", "#fd8d3c", "#a63603"])
    gb = np.array([floor, split, ceil], dtype=float)
    se = np.array([0.0, 0.5, 1.0], dtype=float)
    lb = np.log10(gb)

    def fwd(v):
        v = np.asarray(v, dtype=float)
        out = np.full(v.shape, np.nan, dtype=float)
        f = np.isfinite(v)
        if f.any():
            lv = np.log10(np.clip(v[f], gb[0], gb[-1]))
            seg = np.empty_like(lv)
            for i in range(len(gb) - 1):
                lo, hi = lb[i], lb[i + 1]; last = (i == len(gb) - 2)
                m = (lv >= lo) & (lv <= hi if last else lv < hi)
                seg[m] = se[i] + (lv[m] - lo) / (hi - lo) * (se[i + 1] - se[i])
            out[f] = seg
        return out

    def inv(t):
        t = np.asarray(t, dtype=float)
        out = np.full(t.shape, np.nan, dtype=float)
        f = np.isfinite(t)
        if f.any():
            tc = np.clip(t[f], 0.0, 1.0)
            lv = np.empty_like(tc)
            for i in range(len(se) - 1):
                lo, hi = se[i], se[i + 1]; last = (i == len(se) - 2)
                m = (tc >= lo) & (tc <= hi if last else tc < hi)
                lv[m] = lb[i] + (tc[m] - lo) / (hi - lo) * (lb[i + 1] - lb[i])
            out[f] = np.power(10.0, lv)
        return out

    norm = mcolors.FuncNorm((fwd, inv), vmin=floor, vmax=ceil)
    cmap = mcolors.LinearSegmentedColormap.from_list("low_high", [
        (0.00, low_ramp[0]), (0.25, low_ramp[1]), (0.50, low_ramp[2]),
        (0.50, high_ramp[0]), (0.75, high_ramp[1]), (1.00, high_ramp[2]),
    ])
    cmap.set_bad(globals().get("NO_HIT_COLOR", "#999DA0"))
    return cmap, norm, floor, ceil


_PANEL_CACHE = {}


def dyn_samples():
    """DYN samples present on all three platforms."""
    def coh(c):
        s = set()
        for tool, db in SELECTED_DB_PER_TOOL.items():
            f = cache_tsv_path(c, _TOOL_NAME_IN_CACHE[tool], db, "species")
            if f.exists():
                s |= set(pd.read_csv(f, sep="\t", usecols=["sample_id_core"]).sample_id_core.astype(str))
        return s
    return sorted(set.intersection(*[coh(c) for c in PANEL_COHORTS]))


def panel_data(rank, sample):
    """(order, panels, keyname) for one sample. panels[cohort] = taxa x tool_db FULL matrix
    (sub-threshold values kept; NaN = not reported). order = union of per-cohort list-all taxa."""
    if (rank, sample) in _PANEL_CACHE:
        return _PANEL_CACHE[(rank, sample)]
    keyname, panels, sel = {}, {}, set()
    for cohort in PANEL_COHORTS:
        wl, sc = build_taxa_selection(cohort, sample, SELECTED_DB_PER_TOOL, [rank],
                                      k=CONSENSUS_MIN_SUPPORT, min_abund=PRESENCE_MIN_ABUND)
        sel |= set(wl[rank])
        for k, n in zip(sc[rank]["key"], sc[rank]["name"]):
            keyname.setdefault(k, n)
        td = {f"{tool}_{db}": _load_from_cache(cohort, tool, db, rank, sample)
              for tool, db in SELECTED_DB_PER_TOOL.items()}
        td = {k: td[k] for k in SELECTED_TOOLDBS if k in td}
        panels[cohort] = create_abundance_matrix(td).reindex(columns=SELECTED_TOOLDBS)
    order = sorted(sel)
    panels = {c: panels[c].reindex(index=order) for c in PANEL_COHORTS}
    mx = pd.concat([panels[c].max(axis=1) for c in PANEL_COHORTS], axis=1).max(axis=1).fillna(0.0)
    order = mx.sort_values(ascending=False).index.tolist()
    panels = {c: panels[c].reindex(index=order) for c in PANEL_COHORTS}
    _PANEL_CACHE[(rank, sample)] = (order, panels, keyname)
    return _PANEL_CACHE[(rank, sample)]


def topk_union_rank(panels, order, per_tool_k=PER_TOOL_TOPK):
    """Row order: primary = 'support' (# of the 18 libraries placing the taxon in their own
    top-K); tie-break = detection breadth (# libraries >= MIN_ABUNDANCE), then mean abundance.
    Keeps broadly-agreed taxa on top and sinks single-library abundance spikes (the old maxab
    tie-break floated one-tool artefacts like Buchnera/Phytobacter to the top)."""
    support = pd.Series(0, index=order, dtype=int)
    for c in PANEL_COHORTS:
        for col in SELECTED_TOOLDBS:
            s = panels[c][col]; s = s[s > 0]
            if len(s):
                support.loc[s.nlargest(per_tool_k).index] += 1
    wide = pd.concat([panels[c].reindex(index=order) for c in PANEL_COHORTS], axis=1)
    n_detected = (wide >= MIN_ABUNDANCE).sum(axis=1)
    mean_ab = wide.fillna(0.0).mean(axis=1)
    maxab = wide.max(axis=1).fillna(0.0)
    return pd.DataFrame({"support": support, "n_detected": n_detected,
                         "mean": mean_ab, "maxab": maxab}).sort_values(
        ["support", "n_detected", "mean"], ascending=[False, False, False])


def render_panels(order, panels, keyname, output_path, row_h=ROW_H, max_h=LETTER_H,
                  cbar_label="Abundance (%)", cut_line=None, absent_nan_only=False,
                  title=None, show=True):
    """Letter-size portrait heatmap: taxa x tools, one panel per platform.
    coloured >= MIN_ABUNDANCE; grey = reported but below it; grey + x = not reported / 0.
    Colourbar + no-hit key are VERTICAL in the right margin (saves horizontal space)."""
    cmap, norm, floor, ceil = _two_range_scale()
    tool_names = [t.replace("_unified", "").replace("_default", "") for t in SELECTED_TOOLDBS]
    n_rows, n_cols, n_pan = len(order), len(SELECTED_TOOLDBS), len(PANEL_COHORTS)
    fig_h = max(3.5, n_rows * row_h + OVERHEAD_H)
    if max_h:
        fig_h = min(max_h, fig_h)
    fig, axes = plt.subplots(1, n_pan, sharey=True, figsize=(LETTER_W, fig_h), layout="constrained")
    if n_pan == 1:
        axes = [axes]
    im = None
    for j, cohort in enumerate(PANEL_COHORTS):
        M = panels[cohort].reindex(index=order).values.astype(float)
        colour = np.where(M >= floor, M, np.nan)
        absent = np.isnan(M) if absent_nan_only else (np.isnan(M) | (M == 0))
        ax = axes[j]
        im = ax.imshow(np.ma.masked_invalid(colour), aspect="auto", cmap=cmap, norm=norm,
                       interpolation="nearest")
        ax.set_facecolor(NO_HIT_COLOR)
        rr, cc = np.where(absent)
        ax.scatter(cc, rr, marker="x", s=7, color="#4d4d4d", linewidths=0.5, zorder=4)
        ax.set_title(PANEL_LABELS[cohort], fontsize=8.5, fontweight="bold", pad=3)
        ax.set_xticks(np.arange(n_cols)); ax.set_xticklabels(tool_names, rotation=90, fontsize=6.5)
        ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
        ax.grid(which="minor", color="white", linewidth=0.3); ax.tick_params(which="both", length=0)
        for s in ax.spines.values():
            s.set_edgecolor("#bbbbbb")
    if cut_line and 0 < cut_line < n_rows:
        for ax in axes:
            ax.axhline(cut_line - 0.5, color="black", lw=1.4, ls="--")
    axes[0].set_yticks(np.arange(n_rows))
    axes[0].set_yticklabels([keyname.get(k, k) for k in order], fontsize=6, fontstyle="italic")
    if title:
        fig.suptitle(title, fontsize=9, fontweight="bold")

    # Colourbar + key are sized in ABSOLUTE INCHES (converted to figure fractions), so they
    # stay the same physical size on a 8-in top-K figure and a 34-in uncut "complete" figure.
    CBAR_IN, KEY_IN, GAP_IN = 3.0, 1.4, 0.35
    cbar = fig.colorbar(im, ax=axes, orientation="vertical", location="right",
                        shrink=float(np.clip(CBAR_IN / fig_h, 0.04, 0.6)),
                        aspect=22, pad=0.012, anchor=(0.0, 1.0))
    # always label the low/high split (ABUND_SPLIT) so the colour change is explicit;
    # 3 is dropped because it crowds the 5 tick on a log scale
    _split = float(globals().get("ABUND_SPLIT", 5.0))
    ticks = sorted({t for t in [0.1, 0.3, 1, 10, 30, 100] if floor <= t <= ceil} | {_split})
    cbar.set_ticks(ticks); cbar.set_ticklabels([f"{t:g}" for t in ticks])
    cbar.ax.tick_params(labelsize=6); cbar.set_label(cbar_label, fontsize=7)
    cbar.ax.axhline(0.5, color="black", linewidth=1.0)

    fig.canvas.draw()
    pos = cbar.ax.get_position()
    strip_h = float(np.clip(KEY_IN / fig_h, 0.02, 0.30))     # constant physical height
    lax = fig.add_axes([pos.x0, max(0.01, pos.y0 - strip_h - GAP_IN / fig_h), pos.width, strip_h])
    lax.set_xlim(0, 1); lax.set_ylim(0, 2); lax.axis("off")   # rotated labels do not collide
    for y0, mark, lab in [(1.25, False, f"<{MIN_ABUNDANCE}%"),
                          (0.25, True, "not reported")]:
        lax.add_patch(Rectangle((0, y0), 1.0, 0.5, facecolor=NO_HIT_COLOR,
                                edgecolor="black", lw=0.4, clip_on=False))
        if mark:
            lax.plot([0.5], [y0 + 0.25], marker="x", ms=2.8, color="#4d4d4d", mew=0.7, clip_on=False)
        lax.text(1.7, y0 + 0.25, lab, rotation=90, va="center", ha="center",
                 fontsize=5, clip_on=False)
    save_fig(fig, output_path, SAVE, dpi=FIG_DPI, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)


def panels_table(order, panels, keyname):
    """Long table: taxon x platform x tool abundance (for the per-sample table set)."""
    rows = []
    for c in PANEL_COHORTS:
        m = panels[c].reindex(index=order)
        for k in order:
            for col in SELECTED_TOOLDBS:
                rows.append({"taxon": keyname.get(k, k), "key": k, "platform": c,
                             "tool_db": col, "abundance": m.loc[k, col]})
    return pd.DataFrame(rows)


## Per-sample figure/table sets

In [ ]:
# Per-sample figure/table sets -> <TS>/dyn-heatmap/<Sample>/{figures,tables}
#   complete : every reportable taxon, ranked by top-K/library support (dashed = union cut)
#   topk     : the union-of-top-K rows only (manuscript size)
# ALL samples are written to disk; only DISPLAY_SAMPLE is rendered inline in the notebook.
DISPLAY_SAMPLE = "DYN_0021_D03"     # <- change to display a different sample's set

# Shared cell sizing + panel letters for every figure this notebook writes (also used by the
# min-tools cell below). Narrower columns (LETTER_W) and thinner rows (MANUSCRIPT_ROW_H) than
# the exploratory defaults; species -> (a), genus -> (b). All titles carry the >=0.1% floor.
MANUSCRIPT_FIG_W = 5.0
MANUSCRIPT_ROW_H = 0.09
LETTER_W = MANUSCRIPT_FIG_W
PANEL_LETTER = {"species": "a", "genus": "b"}

SAMPLES_ALL = dyn_samples()
print("samples:", SAMPLES_ALL, "| displaying:", DISPLAY_SAMPLE)

for sample in SAMPLES_ALL:
    figs, tabs = sample_dirs(sample)
    show = (sample == DISPLAY_SAMPLE)
    for rank in RANKS:
        letter = PANEL_LETTER.get(rank, "")
        prefix = f"({letter}) " if letter else ""
        order, panels, keyname = panel_data(rank, sample)
        ordf = topk_union_rank(panels, order)
        order_ranked = ordf.index.tolist()
        n_keep = int((ordf["support"] >= 1).sum())
        order_cut = [k for k in order_ranked if ordf.loc[k, "support"] >= 1]

        render_panels(order_ranked, panels, keyname,
                      figs / f"{sample}_{rank}_heatmap_complete.{FIG_FORMAT}",
                      row_h=MANUSCRIPT_ROW_H, max_h=None, cut_line=n_keep, show=show,
                      title=f"{prefix}{sample} · {rank} · ≥{MIN_ABUNDANCE}% · all taxa "
                            f"(dashed = top-{PER_TOOL_TOPK} cut)")
        render_panels(order_cut, panels, keyname,
                      figs / f"{sample}_{rank}_heatmap_topk{PER_TOOL_TOPK}.{FIG_FORMAT}",
                      row_h=MANUSCRIPT_ROW_H, show=show,
                      title=f"{prefix}{sample} · {rank} · ≥{MIN_ABUNDANCE}% · top-{PER_TOOL_TOPK} union")

        t1 = ordf.assign(taxon=[keyname.get(k, k) for k in ordf.index]).reset_index(names="key")
        save_csv(t1[["taxon", "key", "support", "n_detected", "mean", "maxab"]],
                 tabs / f"{sample}_{rank}_topk{PER_TOOL_TOPK}_support.tsv", SAVE, sep="\t", index=False)
        save_csv(panels_table(order_ranked, panels, keyname),
                 tabs / f"{sample}_{rank}_abundance_long.tsv", SAVE, sep="\t", index=False)
        print(f"  {sample:14} {rank:8}: {len(order_ranked)} taxa, top-{PER_TOOL_TOPK} union = {len(order_cut)} rows")
print("\nPer-sample sets written ->", OUT_BASE)


## Rows by multi-tool agreement (≥2 tools per library, ≥0.1%)

Row rule for the manuscript heatmaps (Fig. 7 + S4-S9): keep taxa corroborated by at least `MIN_TOOL_AGREEMENT` of the 6 profilers at ≥0.1% **on at least one single library** (per-library agreement, not pooled across platforms). Tool-agnostic — removes single-method artefacts symmetrically. Species titles get an `(a)` prefix and genus titles `(b)`; files are tagged `min{MIN_TOOL_AGREEMENT}tools` (currently `min2tools`).

In [ ]:
# === Rows = taxa detected by >= N tools ON ANY ONE LIBRARY (per-library agreement) ===
# A taxon is kept if, on at least one of the three libraries, >= MIN_TOOL_AGREEMENT of the 6
# profilers report it at >= MIN_ABUNDANCE (0.1%). Per-library, not pooled: the tools must concur
# within the same library, so cross-library coincidences do not count. Ordered by agreement,
# then mean abundance.
MIN_TOOL_AGREEMENT = 2

# Filename tag for the manuscript figure/table files (Fig. 7 + S4-S9), derived from the rule so
# it tracks MIN_TOOL_AGREEMENT automatically (currently "min2tools").
FILE_TAG = f"min{MIN_TOOL_AGREEMENT}tools"

# Cell sizing + panel letters are defined in the per-sample cell above (MANUSCRIPT_FIG_W,
# MANUSCRIPT_ROW_H, LETTER_W, PANEL_LETTER) and reused here so every figure matches.


def tool_agreement_counts(panels, order):
    """taxon -> max over the three libraries of (# tools detecting it at >= MIN_ABUNDANCE on
    that library). Per-library agreement, not pooled across platforms."""
    per_lib = [(panels[c].reindex(index=order) >= MIN_ABUNDANCE).sum(axis=1)
               for c in PANEL_COHORTS]
    return pd.concat(per_lib, axis=1).max(axis=1)


def tool_agreement_order(panels, order, min_tools=MIN_TOOL_AGREEMENT):
    n_tools = tool_agreement_counts(panels, order)
    wide = pd.concat([panels[c].reindex(index=order) for c in PANEL_COHORTS], axis=1)
    df = pd.DataFrame({"n_tools": n_tools,
                       "mean": wide.fillna(0.0).mean(axis=1),
                       "maxab": wide.max(axis=1).fillna(0.0)})
    return df[df.n_tools >= int(min_tools)].sort_values(["n_tools", "mean"], ascending=[False, False])


print("\n" + "=" * 60)
print(f"HEATMAPS — taxa detected by >= {MIN_TOOL_AGREEMENT} tools per library at >= {MIN_ABUNDANCE}%")
print("=" * 60)
for sample in SAMPLES_ALL:
    figs, tabs = sample_dirs(sample)
    show = (sample == DISPLAY_SAMPLE)
    for rank in RANKS:
        order, panels, keyname = panel_data(rank, sample)
        ordf = tool_agreement_order(panels, order)
        rows = ordf.index.tolist()
        letter = PANEL_LETTER.get(rank, "")
        prefix = f"({letter}) " if letter else ""
        render_panels(rows, panels, keyname,
                      figs / f"{sample}_{rank}_heatmap_{FILE_TAG}.{FIG_FORMAT}",
                      row_h=MANUSCRIPT_ROW_H, show=show,
                      title=f"{prefix}{sample} · {rank} · ≥{MIN_ABUNDANCE}% · ≥{MIN_TOOL_AGREEMENT} tools")
        t = ordf.assign(taxon=[keyname.get(k, k) for k in ordf.index]).reset_index(names="key")
        save_csv(t[["taxon", "key", "n_tools", "mean", "maxab"]],
                 tabs / f"{sample}_{rank}_{FILE_TAG}.tsv", SAVE, sep="\t", index=False)
        print(f"  {sample:14} {rank:8}: {len(rows)} taxa")
print("\nDone ->", OUT_BASE)
